# 07 地震 · 时空格局

本阶段只用本地主样本，做时间和空间分布，**不使用 USGS、板块边界、活动断层和 CMT**，也不做中国分省图。图写入本文件夹下的 `figures/`，不覆盖已有升级图和旧可视化本。


## STEP 1 · 路径

METHOD: 用 `pathlib.Path` 定位 `07_analysis`。优先桌面项目文件夹，若不存在则回退到当前工作目录或箱子上的同名目录。主样本为 `07_earthquake_analysis_sample.csv`。底图在上一级 `mapdata/` 中查找（Natural Earth 50m 陆地）。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
import matplotlib.font_manager as fm
import geopandas as gpd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pyproj import Transformer
from shapely.geometry import box, LineString

CANDIDATES = [
    Path(r"C:\Users\WHB\Desktop\DPW I\07_analysis"),
    Path.cwd(),
    Path("/workspace/07_analysis"),
]

ROOT = None
for cand in CANDIDATES:
    if (cand / "07_earthquake_analysis_sample.csv").exists():
        ROOT = cand
        break
if ROOT is None:
    raise FileNotFoundError("找不到 07_earthquake_analysis_sample.csv")

MAP_CANDIDATES = [
    ROOT.parent / "mapdata",
    Path(r"C:\Users\WHB\Desktop\DPW I\mapdata"),
    Path("/workspace/dpw1_report/mapdata"),
]
MAPDATA = next(
    (p for p in MAP_CANDIDATES if (p / "ne_50m_land" / "ne_50m_land.shp").exists()),
    None,
)
if MAPDATA is None:
    raise FileNotFoundError("找不到 Natural Earth 50m 陆地底图 mapdata/ne_50m_land")

SAMPLE_IN = ROOT / "07_earthquake_analysis_sample.csv"
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("主样本:", SAMPLE_IN)
print("MAPDATA:", MAPDATA)
print("FIGURES:", FIGURES)


## STEP 2 · 读取主样本并生成分析字段

METHOD: 读取本地主样本。`Date_parsed` 按 UTC、ISO8601 解析，年份取自 `Date_parsed`，十年为 `year // 10 * 10`（1960s 从 1965 年起，2010s 到 2016 年止）。震级分箱为右开区间：5.5–5.9、6.0–6.9、7.0–7.9、≥8.0。深度：小于 70 公里为浅源，70–300 公里为中源，大于 300 公里为深源；深度缺失先记为空，不在这一步删除。不删行。


In [ ]:
def coerce_bool(s):
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)
    return s.astype(str).str.lower().isin(["true", "1", "yes"])

df = pd.read_csv(SAMPLE_IN)
n0 = len(df)
df["Date_parsed"] = pd.to_datetime(df["Date_parsed"], format="ISO8601", utc=True, errors="coerce")
for col in ["flag_coord_invalid", "flag_mag_outlier", "flag_depth_negative", "in_china"]:
    if col in df.columns:
        df[col] = coerce_bool(df[col])

df["year"] = df["Date_parsed"].dt.year
df["decade"] = (df["year"] // 10) * 10
df["decade_label"] = df["decade"].astype("Int64").astype(str) + "s"

MAG_BINS = [5.5, 6.0, 7.0, 8.0, 10.0]
MAG_LABELS = ["5.5–5.9", "6.0–6.9", "7.0–7.9", "≥8.0"]
df["mag_bin"] = pd.cut(
    df["Magnitude"], bins=MAG_BINS, right=False, labels=MAG_LABELS, include_lowest=True
)

def depth_class(d):
    if pd.isna(d):
        return pd.NA
    if d < 70:
        return "浅源 <70 km"
    if d <= 300:
        return "中源 70–300 km"
    return "深源 >300 km"

df["depth_class"] = df["Depth"].map(depth_class)
print("读取行数:", n0, "处理后行数:", len(df), "变化量:", len(df) - n0)
print("Date_parsed 失败:", int(df["Date_parsed"].isna().sum()),
      "| 年份缺失:", int(df["year"].isna().sum()))
print("年份范围:", int(df["year"].min()), "–", int(df["year"].max()))
print("震级分箱缺失:", int(df["mag_bin"].isna().sum()))
print("深度类别缺失:", int(df["depth_class"].isna().sum()))
df.head(3)


## STEP 3 · 清点分析口径

METHOD: 打印震级分箱、深度类别和十年频次。空间图排除 `flag_coord_invalid` 或经纬度缺失的行；震级统计排除 `flag_mag_outlier`；深度图再排除深度缺失。疑似重复全部保留。


In [ ]:
spatial = df.loc[~df["flag_coord_invalid"] & df["Latitude"].notna() & df["Longitude"].notna()].copy()
mag_ok = df.loc[~df["flag_mag_outlier"]].copy()
depth_ok = spatial.loc[spatial["depth_class"].notna()].copy()
m7 = spatial.loc[spatial["Magnitude"] >= 7].copy()

print("主样本:", len(df))
print("可用于空间图:", len(spatial), "| 排除坐标无效或缺失:", len(df) - len(spatial))
print("可用于震级统计:", len(mag_ok), "| 排除震级异常:", len(df) - len(mag_ok))
print("可用于深度图:", len(depth_ok), "| 空间图中深度缺失:", len(spatial) - len(depth_ok))
print("M≥7:", len(m7))
print()
print("震级分箱:")
print(mag_ok["mag_bin"].value_counts(sort=False))
print()
print("深度类别:")
print(depth_ok["depth_class"].value_counts())
print()
print("十年频次:")
print(df.groupby("decade_label", dropna=False).size())


## STEP 4 · 配色、字体与投影函数

METHOD: 地图沿用已有升级方案的深棕金配色，等积地球投影、中央经线 150°E；投影前沿对跖经线（−30°）切开多边形，避免条带。本本底图只用 Natural Earth 50m 陆地和经纬网，不叠板块边界。中文字体在 Linux 上用 Noto Sans CJK，在 Windows 上用微软雅黑。图以 300 dpi 写入 `figures/`。


In [ ]:
WORLD_CRS = "+proj=eqearth +lon_0=150 +datum=WGS84 +units=m +no_defs"
LON0 = 150.0
OCEAN = "#100C08"
LAND = "#1C1510"
GOLD = "#D5AA6D"
GOLD_SOFT = "#E6C48A"
CREAM = "#FCF7EE"
MUTED = "#C9B79A"
POINT_FACE = "#F0B429"
POINT_EDGE = "#FFF1C8"
CMAP_DENSITY = mcolors.LinearSegmentedColormap.from_list(
    "eq_density",
    ["#4A3018", "#8B5A28", "#C4923A", "#E0B020", "#F4D060", "#FFE9A0"],
)
MAG_COLORS = {
    "5.5–5.9": "#7A4E22",
    "6.0–6.9": "#C4923A",
    "7.0–7.9": "#E0B020",
    "≥8.0": "#FFE9A0",
}
DEPTH_ORDER = ["浅源 <70 km", "中源 70–300 km", "深源 >300 km"]
DEPTH_COLORS = {
    "浅源 <70 km": "#F0B429",
    "中源 70–300 km": "#E07A3D",
    "深源 >300 km": "#F6E3B0",
}

def find_cjk_fonts():
    regular_cands = [
        Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
        Path(r"C:\Windows\Fonts\msyh.ttc"),
        Path(r"C:\Windows\Fonts\msyh.ttf"),
        Path(r"C:\Windows\Fonts\simhei.ttf"),
        Path(r"C:\Windows\Fonts\simsun.ttc"),
    ]
    bold_cands = [
        Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc"),
        Path(r"C:\Windows\Fonts\msyhbd.ttc"),
        Path(r"C:\Windows\Fonts\msyhbd.ttf"),
    ]
    regular = next((p for p in regular_cands if p.exists()), None)
    bold = next((p for p in bold_cands if p.exists()), regular)
    if regular is None:
        for name in ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Source Han Sans SC"]:
            for f in fm.fontManager.ttflist:
                if name.lower() in f.name.lower():
                    regular = Path(f.fname)
                    break
            if regular:
                break
    if regular is None:
        raise RuntimeError("未找到中文字体")
    return (
        fm.FontProperties(fname=str(regular)),
        fm.FontProperties(fname=str(bold if bold else regular)),
    )

FONT, FONT_BOLD = find_cjk_fonts()
HALO = [pe.withStroke(linewidth=2.4, foreground=OCEAN)]

def style_text(t, bold=False, size=None, color=CREAM):
    t.set_fontproperties(FONT_BOLD if bold else FONT)
    t.set_color(color)
    if size is not None:
        t.set_fontsize(size)
    t.set_path_effects(HALO)
    return t

def cut_at_seam(gdf, lon0=LON0, lat_limit=89.9):
    gdf = gdf.copy()
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    gdf["geometry"] = gdf.geometry.make_valid()
    seam = lon0 - 180.0
    left = box(-180.0, -lat_limit, seam, lat_limit)
    right = box(seam, -lat_limit, 180.0, lat_limit)
    parts = pd.concat([gdf.clip(left), gdf.clip(right)], ignore_index=True)
    parts = parts[parts.geometry.notna() & ~parts.geometry.is_empty]
    return gpd.GeoDataFrame(parts, crs="EPSG:4326")

def to_world(gdf):
    return cut_at_seam(gdf).to_crs(WORLD_CRS)

def project_xy(lons, lats, crs=WORLD_CRS):
    tf = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
    return tf.transform(np.asarray(lons, dtype=float), np.asarray(lats, dtype=float))

def make_graticule(dlon=30, dlat=30):
    lines = []
    for lon in range(-180, 180, dlon):
        lines.append(LineString([(lon, lat) for lat in range(-80, 81, 2)]))
    for lat in range(-60, 90, dlat):
        lines.append(LineString([(lon, lat) for lon in range(-180, 181, 3)]))
    return gpd.GeoDataFrame(geometry=lines, crs="EPSG:4326")

def restyle_axes(ax):
    ax.set_facecolor(OCEAN)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_color(GOLD)
        spine.set_linewidth(0.9)
    ax.set_aspect("equal")

def clear_xy_labels(ax):
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

def wrap_cjk(text, width=78):
    lines, buf = [], ""
    for ch in text:
        buf += ch
        if len(buf) >= width and ch in "。；，、： ":
            lines.append(buf.strip())
            buf = ""
    if buf.strip():
        lines.append(buf.strip())
    return "\n".join(lines)

def add_caption(fig, text, width=86):
    fig.text(
        0.01, 0.012, wrap_cjk(text, width=width),
        fontproperties=FONT, fontsize=7.6, color=MUTED,
        ha="left", va="bottom", linespacing=1.35,
    )

def style_colorbar(cb, label):
    cb.ax.yaxis.set_tick_params(color=GOLD, labelcolor=CREAM, labelsize=8)
    cb.outline.set_edgecolor(GOLD)
    cb.outline.set_linewidth(0.6)
    cb.set_label(label, fontproperties=FONT, color=CREAM, fontsize=9)
    for t in cb.ax.get_yticklabels():
        t.set_fontproperties(FONT)
        t.set_color(CREAM)

def style_legend(leg):
    frame = leg.get_frame()
    frame.set_facecolor("#1A140F")
    frame.set_edgecolor(GOLD)
    frame.set_linewidth(0.6)
    frame.set_alpha(0.92)
    if leg.get_title():
        leg.get_title().set_fontproperties(FONT)
        leg.get_title().set_color(CREAM)
        leg.get_title().set_fontsize(9)
    for t in leg.get_texts():
        t.set_fontproperties(FONT)
        t.set_color(CREAM)
        t.set_fontsize(8)

def save_fig(fig, path):
    for ax in fig.axes:
        xlab, ylab = str(ax.get_xlabel()), str(ax.get_ylabel())
        if "east" in xlab.lower() or "metre" in xlab.lower() or "meter" in xlab.lower():
            ax.set_xlabel("")
        if "north" in ylab.lower() or "metre" in ylab.lower() or "meter" in ylab.lower():
            ax.set_ylabel("")
    fig.savefig(
        path, dpi=300, facecolor=fig.get_facecolor(), edgecolor="none",
        bbox_inches="tight", pad_inches=0.18,
    )
    plt.close(fig)
    print("wrote", path, "bytes", Path(path).stat().st_size)

def mag_size(mag, base=40.0, scale=70.0, floor=7.0):
    mag = np.asarray(mag, dtype=float)
    return base + scale * np.clip(mag - floor, 0, None) ** 2

def style_time_axes(ax):
    ax.set_facecolor(OCEAN)
    ax.tick_params(colors=CREAM, labelcolor=CREAM)
    for spine in ax.spines.values():
        spine.set_color(GOLD)
        spine.set_linewidth(0.8)
    ax.yaxis.grid(True, color=GOLD, alpha=0.18, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    for t in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        t.set_fontproperties(FONT)
        t.set_color(CREAM)

print("配色与投影函数已就绪。字体:", FONT.get_name())


## STEP 5 · 年度频次（按震级分箱堆叠）

METHOD: 以年份为横轴，按震级分箱堆叠柱状图。使用可用于震级统计的样本。图保存为 `figures/07_st_annual_mag.png`。


In [ ]:
years = list(range(int(mag_ok["year"].min()), int(mag_ok["year"].max()) + 1))
annual = (
    mag_ok.groupby(["year", "mag_bin"], observed=False).size()
    .unstack(fill_value=0)
    .reindex(index=years, fill_value=0)
    .reindex(columns=MAG_LABELS, fill_value=0)
)

fig = plt.figure(figsize=(16.2, 6.4), facecolor=OCEAN)
ax = fig.add_subplot(111)
fig.subplots_adjust(left=0.07, right=0.98, top=0.88, bottom=0.18)
style_time_axes(ax)
bottom = np.zeros(len(annual))
x = annual.index.to_numpy()
for lab in MAG_LABELS:
    vals = annual[lab].to_numpy()
    ax.bar(x, vals, bottom=bottom, color=MAG_COLORS[lab], width=0.86, linewidth=0, label=lab, zorder=3)
    bottom += vals
ax.set_xlim(x.min() - 0.8, x.max() + 0.8)
ax.set_ylabel("地震次数", fontproperties=FONT, color=CREAM, fontsize=10)
title = ax.set_title("本地构造地震年度频次（按震级分箱）", fontproperties=FONT_BOLD, fontsize=16, color=CREAM, pad=10)
title.set_path_effects(HALO)
leg = ax.legend(title="震级", loc="upper left", framealpha=0.92, ncol=4)
style_legend(leg)
add_caption(
    fig,
    f"数据：本地主样本构造地震（1965–2016，M≥5.5，n={len(mag_ok):,}）。"
    "分箱为 5.5–5.9、6.0–6.9、7.0–7.9、≥8.0（右开区间，6.0 计入 6.0–6.9）。"
    "疑似重复全部保留。本图不做完整性校正。",
    width=88,
)
ANNUAL_FIG = FIGURES / "07_st_annual_mag.png"
save_fig(fig, ANNUAL_FIG)


## STEP 6 · 十年频次（按震级分箱堆叠）

METHOD: 按十年汇总，同样按震级分箱堆叠。图中注明 1960s 仅含 1965–1969 年、2010s 仅含 2010–2016 年。保存为 `figures/07_st_decade_mag.png`。


In [ ]:
decade_order = sorted(mag_ok["decade"].dropna().unique())
decade_labels = [f"{int(d)}s" for d in decade_order]
decadal = (
    mag_ok.groupby(["decade", "mag_bin"], observed=False).size()
    .unstack(fill_value=0)
    .reindex(index=decade_order, fill_value=0)
    .reindex(columns=MAG_LABELS, fill_value=0)
)

fig = plt.figure(figsize=(12.0, 6.4), facecolor=OCEAN)
ax = fig.add_subplot(111)
fig.subplots_adjust(left=0.10, right=0.98, top=0.88, bottom=0.18)
style_time_axes(ax)
x = np.arange(len(decadal))
bottom = np.zeros(len(decadal))
for lab in MAG_LABELS:
    vals = decadal[lab].to_numpy()
    ax.bar(x, vals, bottom=bottom, color=MAG_COLORS[lab], width=0.66, linewidth=0, label=lab, zorder=3)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(decade_labels)
ax.set_ylabel("地震次数", fontproperties=FONT, color=CREAM, fontsize=10)
title = ax.set_title("本地构造地震十年频次（按震级分箱）", fontproperties=FONT_BOLD, fontsize=16, color=CREAM, pad=10)
title.set_path_effects(HALO)
leg = ax.legend(title="震级", loc="upper left", framealpha=0.92, ncol=2)
style_legend(leg)
add_caption(
    fig,
    f"数据：本地主样本构造地震（n={len(mag_ok):,}）。"
    "1960s 仅含 1965–1969 年，2010s 仅含 2010–2016 年，这两个十年的频次不可与完整十年直接比较。"
    "分箱规则与年度图相同。",
    width=80,
)
DECADE_FIG = FIGURES / "07_st_decade_mag.png"
save_fig(fig, DECADE_FIG)


## STEP 7 · 载入底图并投影

METHOD: 读取 Natural Earth 50m 陆地，生成 30° 经纬网，沿对跖经线切开后投影到 Equal Earth（150°E）。不读取板块边界。


In [ ]:
land50 = gpd.read_file(MAPDATA / "ne_50m_land" / "ne_50m_land.shp")
print("projecting world layers ...")
land_w = to_world(land50)
graticule_w = to_world(make_graticule())
print("land50", land50.shape, "| projected land bounds:", land_w.total_bounds)

def draw_world_basemap(ax, land_w, graticule_w=None):
    if graticule_w is not None and not graticule_w.empty:
        graticule_w.plot(ax=ax, color=GOLD, linewidth=0.25, alpha=0.18, zorder=0.5)
    land_w.plot(ax=ax, color=LAND, edgecolor=GOLD, linewidth=0.28, zorder=1)

def fit_world_view(ax, land_w):
    bounds = land_w.total_bounds
    pad_x = (bounds[2] - bounds[0]) * 0.015
    pad_y = (bounds[3] - bounds[1]) * 0.02
    ax.set_xlim(bounds[0] - pad_x, bounds[2] + pad_x)
    ax.set_ylim(bounds[1] - pad_y, bounds[3] + pad_y)


## STEP 8 · 全球密度图与 M≥7 分布

METHOD: 底层为全部空间样本的六边形密度（对数色阶），上层只标注 M≥7 震中，符号大小随震级增加。不叠板块边界。保存为 `figures/07_st_global_density_m7.png`。


In [ ]:
fig = plt.figure(figsize=(16.2, 9.05), facecolor=OCEAN)
ax = fig.add_subplot(111)
restyle_axes(ax)
fig.subplots_adjust(left=0.03, right=0.90, top=0.90, bottom=0.14)
draw_world_basemap(ax, land_w, graticule_w)

x, y = project_xy(spatial["Longitude"], spatial["Latitude"])
hb = ax.hexbin(x, y, gridsize=92, mincnt=1, cmap=CMAP_DENSITY, linewidths=0, alpha=0.90, zorder=3)
counts = hb.get_array()
vmax = max(8.0, float(np.nanmax(counts)) if len(counts) else 8.0)
hb.set_norm(mcolors.LogNorm(vmin=1, vmax=vmax))
hb.set_cmap(CMAP_DENSITY)

bx, by = project_xy(m7["Longitude"], m7["Latitude"])
sizes = mag_size(m7["Magnitude"])
ax.scatter(bx, by, s=sizes * 1.85, c="#E8A030", alpha=0.16, linewidths=0, zorder=5)
ax.scatter(bx, by, s=sizes, facecolors=POINT_FACE, edgecolors=POINT_EDGE,
           linewidths=0.65, alpha=0.92, zorder=6)
fit_world_view(ax, land_w)

cb = fig.colorbar(hb, ax=ax, fraction=0.026, pad=0.015, shrink=0.55)
style_colorbar(cb, "六边形内频次（对数）")
handles = []
for m in (7.0, 8.0, 9.0):
    handles.append(Line2D(
        [0], [0], marker="o", linestyle="none",
        markersize=np.sqrt(mag_size(m)) * 0.55,
        markerfacecolor=POINT_FACE, markeredgecolor=POINT_EDGE,
        markeredgewidth=0.6, label=f"M {m:.1f}",
    ))
leg = ax.legend(handles=handles, title="7 级以上震中", loc="lower left",
                framealpha=0.92, labelspacing=1.05, borderpad=0.7)
style_legend(leg)
title = ax.set_title("本地构造地震震中分布（密度 + 7 级以上）", fontproperties=FONT_BOLD, fontsize=17, color=CREAM, pad=12)
title.set_path_effects(HALO)
clear_xy_labels(ax)
add_caption(
    fig,
    f"数据：本地主样本构造地震（1965–2016，M≥5.5，空间样本 n={len(spatial):,}；其中 M≥7 共 {len(m7):,} 次）。"
    "投影：Equal Earth，中央经线 150°E，避免从 180° 切开环太平洋地震带。"
    "底层为全部事件的六边形密度（对数色阶），上层仅标注 7 级以上震中，符号大小随震级增加。"
    "本图不叠板块边界。底图：Natural Earth 50m。",
    width=86,
)
DENSITY_FIG = FIGURES / "07_st_global_density_m7.png"
save_fig(fig, DENSITY_FIG)


## STEP 9 · 按深度分层的全球分布

METHOD: 同一投影、同一视野下分三幅：浅源小于 70 公里、中源 70–300 公里、深源大于 300 公里。深度缺失的 3 条只从本图排除。保存为 `figures/07_st_global_depth.png`。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.2, 6.35), facecolor=OCEAN)
fig.subplots_adjust(left=0.02, right=0.99, top=0.86, bottom=0.16, wspace=0.04)
for ax, cls in zip(axes, DEPTH_ORDER):
    restyle_axes(ax)
    draw_world_basemap(ax, land_w, graticule_w)
    sub = depth_ok.loc[depth_ok["depth_class"] == cls]
    sx, sy = project_xy(sub["Longitude"], sub["Latitude"])
    sizes = 6.0 + 8.0 * np.clip(sub["Magnitude"].to_numpy(dtype=float) - 5.5, 0, None)
    ax.scatter(
        sx, sy, s=sizes, facecolors=DEPTH_COLORS[cls], edgecolors=POINT_EDGE,
        linewidths=0.15, alpha=0.78, zorder=4,
    )
    fit_world_view(ax, land_w)
    clear_xy_labels(ax)
    title = ax.set_title(f"{cls}  n={len(sub):,}", fontproperties=FONT_BOLD, fontsize=12, color=CREAM, pad=8)
    title.set_path_effects(HALO)

fig.suptitle("本地构造地震按深度分层的全球分布", fontproperties=FONT_BOLD, fontsize=16, color=CREAM, y=0.97)
add_caption(
    fig,
    f"数据：本地主样本中深度非缺失的构造地震（n={len(depth_ok):,}；另有 {len(spatial) - len(depth_ok)} 条深度缺失只从本图排除）。"
    "浅源 <70 km，中源 70–300 km（含 70 与 300），深源 >300 km。"
    "三幅图使用同一 Equal Earth 投影（中央经线 150°E）和同一视野，符号略随震级增大。底图：Natural Earth 50m。",
    width=90,
)
DEPTH_FIG = FIGURES / "07_st_global_depth.png"
save_fig(fig, DEPTH_FIG)


## STEP 10 · 导出频次表并汇总

METHOD: 把年度、十年频次表写到 `07_analysis/`，不覆盖主样本和精细清洗文件。打印各图路径与关键计数。


In [ ]:
annual_out = annual.copy()
annual_out["total"] = annual_out.sum(axis=1)
ANNUAL_CSV = ROOT / "07_st_annual_counts.csv"
annual_out.to_csv(ANNUAL_CSV)
print("wrote", ANNUAL_CSV)

decade_out = decadal.copy()
decade_out["total"] = decade_out.sum(axis=1)
decade_out.index = [f"{int(i)}s" for i in decade_out.index]
DECADE_CSV = ROOT / "07_st_decade_counts.csv"
decade_out.to_csv(DECADE_CSV)
print("wrote", DECADE_CSV)

print()
print("—— 时空格局小结 ——")
print(f"主样本 {len(df):,} | 空间样本 {len(spatial):,} | 震级统计 {len(mag_ok):,} | 深度图 {len(depth_ok):,}")
print("震级分箱:", mag_ok["mag_bin"].value_counts(sort=False).to_dict())
print("深度类别:", depth_ok["depth_class"].value_counts().to_dict())
print("M≥7:", len(m7), "| 年度合计最小/最大:", int(annual_out["total"].min()), int(annual_out["total"].max()))
print("十年合计:")
print(decade_out["total"].to_string())
print()
for p in [ANNUAL_FIG, DECADE_FIG, DENSITY_FIG, DEPTH_FIG, ANNUAL_CSV, DECADE_CSV]:
    print("输出:", p)


## STEP 11 · 检查输出，确认旧文件未被覆盖

METHOD: 确认四张新图和两份频次表都已写出；并确认没有写入旧的 `figures/07_map_*.png` 或精细清洗文件。


In [ ]:
outputs = [ANNUAL_FIG, DECADE_FIG, DENSITY_FIG, DEPTH_FIG, ANNUAL_CSV, DECADE_CSV]
for p in outputs:
    print(p.name, "exists:" , p.exists(), "bytes:", p.stat().st_size if p.exists() else 0)

old_map_names = [
    "07_map_global_density.png",
    "07_map_china_province.png",
    "07_map_country_choropleth.png",
    "07_map_top30.png",
]
written_names = {p.name for p in outputs}
print("未使用旧升级图文件名:", written_names.isdisjoint(old_map_names))
print("主样本仍在:", SAMPLE_IN.exists())
print("输出目录是本文件夹 figures/:", FIGURES.resolve() == (ROOT / "figures").resolve())
